In [ ]:
import json
from pathlib import Path

import pandas as pd

import re

from collections import Counter

from transformers import CLIPModel, CLIPProcessor

import torch
import torch.nn.functional as F
from PIL import Image
import os
from tqdm import tqdm

import numpy as np

In [133]:
DATA_DIR  = Path("../Dataset/JSON")
IMAGE_DIR = Path("../Dataset/dataset_image")

IMAGE_DATA_DIR = Path("../Dataset/dataset_image")

OUTPUT_DIR = Path("../Output")

MODEL_NAME="openai/clip-vit-large-patch14-336"

DEVICE = "cpu"

PATCH_SIZE = 14

In [134]:
model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 35727.64it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [135]:
with open(DATA_DIR / "test.json", encoding="utf-8") as f:
    test_data = json.load(f)

with open(DATA_DIR / "valid.json", encoding="utf-8") as f:
    val_data = json.load(f)

with open(DATA_DIR / "train.json", encoding="utf-8") as f:
    train_data = json.load(f)

df_test = pd.DataFrame(test_data)
df_val = pd.DataFrame(val_data)
df_train = pd.DataFrame(train_data)

print(f"Train Data Length: {len(df_train)}")
print(f"Validation Data Length: {len(df_val)}")
print(f"Test Data Length: {len(df_test)}")
df_train.head(5)

Train Data Length: 19816
Validation Data Length: 2410
Test Data Length: 2409


,image_id,text,label
0,840006160660983809,<user> thanks for showing up for our appointme...,1
1,908913372199915520,haha .,1
2,916496521406726145,i love waiting <num> min for a cab - such shor...,1
3,916364004129304576,22 super funny quotes <user>,1
4,853866052589154304,goog morning,1


In [136]:
df_master = pd.concat([df_test, df_train, df_val])
print(len(df_master))

24635


In [137]:
word_list = set()
for text in df_master["text"]:
    words = text.split(" ")
    for word in words:
        text_0 = re.findall(r'<[^>]*>', word)
        if len(text_0) != 0:
            word_list.add(word)

print(word_list)

{'<num>', '<facepalm>', '<>', '<user>', '<<url>'}


In [138]:
tag_counts = Counter()
for text in df_master["text"]:
    for word in str(text).split():
        if re.fullmatch(r'<[^>]*>', word):
            tag_counts[word] += 1

print(tag_counts.most_common())

[('<user>', 8629), ('<num>', 577), ('<>', 2), ('<<url>', 1), ('<facepalm>', 1)]


In [139]:
image_path = IMAGE_DATA_DIR / "682716753374351360.jpg"
print(image_path)
sample_image = Image.open(image_path).convert("RGB")

sample_image.size

..\Dataset\dataset_image\682716753374351360.jpg


(800, 563)

In [140]:
inputs = processor(images=sample_image, return_tensors="pt")
pixel_values = inputs["pixel_values"].to(DEVICE)

print("Shape: ", pixel_values.shape)


Shape:  torch.Size([1, 3, 336, 336])


In [141]:
patches = pixel_values.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
print(patches.shape)
patches = patches.permute(0,2,3,1,4,5)
print(patches.shape)
patches = patches.reshape(1,576,3,14,14)
print(patches.shape)

torch.Size([1, 3, 24, 24, 14, 14])
torch.Size([1, 24, 24, 3, 14, 14])
torch.Size([1, 576, 3, 14, 14])


In [142]:
#List of patches dimana pixel steiap patchesnya sudah di flatten
flat_patches = patches.reshape(1, 576, -1)
print("Flatten Patches: ", flat_patches.shape)

Flatten Patches:  torch.Size([1, 576, 588])


In [143]:
with torch.no_grad():
    vision = model.vision_model
    emb = vision.embeddings

    patch_grid = emb.patch_embedding(pixel_values)
    print(patch_grid.shape)
    patch_embeddings = patch_grid.flatten(2).transpose(1, 2)
    print("patch emb: ", patch_embeddings.shape)
    batch_size = patch_embeddings.shape[0]
    print("emb class: ", emb.class_embedding.shape)
    cls_token = emb.class_embedding.expand(batch_size, 1, -1)
    print(cls_token.shape)

    token_emb = torch.concat([patch_embeddings, cls_token], dim=1)
    print("total token: ", token_emb.shape)

    position_ids = torch.arange(token_emb.shape[1], device=pixel_values.device).unsqueeze(0)
    position_embeddings = emb.position_embedding(position_ids)

    final_embedding = token_emb + position_embeddings
    print("position_embeddings:", position_embeddings.shape)
    print("final_embedding:", final_embedding.shape)

    hidden_states = vision.pre_layrnorm(final_embedding)
    encoder_outputs = vision.encoder(
        inputs_embeds=hidden_states,
        output_hidden_states=True,
        return_dict=True
    )

    visual_features = encoder_outputs.last_hidden_state
    print("visual features shape: ", visual_features.shape)
    cls_output = visual_features[:, 0, :]
    print("cls_output before post norm:", cls_output.shape)

    embv_1024 = vision.post_layernorm(cls_output)
    print("EmbV 1024:", embv_1024.shape)

    embv_768 = model.visual_projection(embv_1024)

    embv_768 = F.normalize(embv_768, p=2, dim=-1)
    print("Final CLIP image embedding:", embv_768.shape)

    print(embv_768)


torch.Size([1, 1024, 24, 24])
patch emb:  torch.Size([1, 576, 1024])
emb class:  torch.Size([1024])
torch.Size([1, 1, 1024])
total token:  torch.Size([1, 577, 1024])
position_embeddings: torch.Size([1, 577, 1024])
final_embedding: torch.Size([1, 577, 1024])
visual features shape:  torch.Size([1, 577, 1024])
cls_output before post norm: torch.Size([1, 1024])
EmbV 1024: torch.Size([1, 1024])
Final CLIP image embedding: torch.Size([1, 768])
tensor([[ 8.5212e-03,  2.1927e-02, -1.8225e-02, -4.3406e-03,  3.2806e-02,
         -7.8914e-05, -2.3420e-03, -1.3715e-02, -1.3956e-02, -5.8520e-03,
          3.0947e-02,  4.8188e-02, -2.1490e-02, -1.5646e-02,  4.6379e-02,
          1.4034e-02,  1.8484e-02, -4.8363e-02, -5.8770e-02,  2.7424e-02,
          3.4497e-02, -4.1968e-03,  9.3062e-03,  1.1130e-02,  1.6198e-02,
         -3.8794e-02,  1.5875e-02,  2.4333e-02,  3.0814e-02, -2.9313e-02,
          6.4053e-02,  4.6869e-02,  6.2911e-02,  4.1793e-02, -1.8274e-02,
          9.7640e-03,  2.5834e-02,  7.66

In [144]:
sample_query_text = "My Dog Step on a Bee"
with torch.no_grad():
    text_inputs = processor(
        text=[sample_query_text],
        return_tensors="pt",
        padding="max_length",
        max_length=77,
        truncation=True
    )

    input_ids = text_inputs["input_ids"].to(DEVICE)
    attention_mask = text_inputs["attention_mask"].to(DEVICE)

    text_outputs = model.text_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True
    )

    text_features = text_outputs.last_hidden_state
    embt = text_outputs.pooler_output

    print("input_ids:", input_ids.shape)
    print("attention_mask:", attention_mask.shape)
    print("text_features:", text_features.shape)
    print("EmbT:", embt.shape)

input_ids: torch.Size([1, 77])
attention_mask: torch.Size([1, 77])
text_features: torch.Size([1, 77, 768])
EmbT: torch.Size([1, 768])


In [145]:
def image_embedder(img_path: str):
    image = Image.open(img_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)

    with torch.no_grad():
        vision = model.vision_model
        emb = vision.embeddings
        
        patch_grid = emb.patch_embedding(pixel_values)
        patch_embeddings = patch_grid.flatten(2).transpose(1, 2)

        batch_size = patch_embeddings.shape[0]
        cls_token = emb.class_embedding.expand(batch_size, 1, -1)

        token_emb = torch.cat([cls_token, patch_embeddings], dim=1)

        position_ids = torch.arange(token_emb.shape[1], device=pixel_values.device).unsqueeze(0)
        position_embeddings = emb.position_embedding(position_ids)

        final_embedding = token_emb + position_embeddings

        hidden_states = vision.pre_layrnorm(final_embedding)

        encoder_outputs = vision.encoder(
            inputs_embeds=hidden_states,
            output_hidden_states=True,
            return_dict=True
        )

        visual_features = encoder_outputs.last_hidden_state

        cls_output = visual_features[:, 0, :]
        embv_1024 = vision.post_layernorm(cls_output)

        embv_768 = model.visual_projection(embv_1024)
        embv_768 = F.normalize(embv_768, p=2, dim=-1)

        return embv_768

In [146]:
def text_embedder(query: str):
    with torch.no_grad():
        text_inputs = processor(
            text=[query],
            return_tensors="pt",
            padding="max_length",
            max_length=77,
            truncation=True
        )

        input_ids = text_inputs["input_ids"].to(DEVICE)
        attention_mask = text_inputs["attention_mask"].to(DEVICE)

        text_outputs = model.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        embt = text_outputs.pooler_output        
        embt = model.text_projection(embt)       
        embt = F.normalize(embt, p=2, dim=-1)

        return embt

In [147]:
image_embedder(IMAGE_DATA_DIR / "682716753374351360.jpg").shape

torch.Size([1, 768])

In [148]:
text_embedder("My Dog Step On A Bee").shape

torch.Size([1, 768])

In [149]:
image_urls = []
for image_name in os.listdir(IMAGE_DATA_DIR):
    full_path = IMAGE_DATA_DIR / image_name
    image_urls.append(full_path)
len(image_urls)

24635

In [150]:
df_train.head(5)

,image_id,text,label
0,840006160660983809,<user> thanks for showing up for our appointme...,1
1,908913372199915520,haha .,1
2,916496521406726145,i love waiting <num> min for a cab - such shor...,1
3,916364004129304576,22 super funny quotes <user>,1
4,853866052589154304,goog morning,1


In [151]:
#df_train = df_train[:10]
df_train = df_train.reset_index(drop=True)

In [152]:
metadata_rows = []
embv_list = []
embt_list = []

for index, row in tqdm(df_train.iterrows(), total=len(df_train)):
    image_full_path = IMAGE_DATA_DIR / f"{str(row['image_id'])}.jpg"
    text = row["text"]
    label = row["label"]

    embt = text_embedder(text)
    embv = image_embedder(image_full_path)
    embv_list.append(embv)
    embt_list.append(embt)

    metadata_rows.append({
        "image_path": str(image_full_path),
        "text": text,
        "label": label
    })

metadata_df = pd.DataFrame(metadata_rows)

embv_array = np.stack(embv_list)
embt_array = np.stack(embt_list)


metadata_df.to_csv(os.path.join(OUTPUT_DIR, "train_metadata.csv"), index=False)
np.save(os.path.join(OUTPUT_DIR, "train_embv.npy"), embv_array)
np.save(os.path.join(OUTPUT_DIR, "train_embt.npy"), embt_array)


100%|██████████| 19816/19816 [6:28:41<00:00,  1.18s/it]  


In [ ]:
metadata_df = pd.read_csv(OUTPUT_DIR / "train_metadata.csv")
train_embv = np.load(OUTPUT_DIR / "train_embv.npy").astype("float32")
train_embt = np.load(OUTPUT_DIR / "train_embt.npy").astype("float32")

In [ ]:
def retrieve_top_k(query_embedding, db_embeddings, metadata_df, top_k=5, exclude_image_id=None):
    query_vec = query_embedding.squeeze(0).detach().cpu().numpy().astype("float32")

    scores = db_embeddings @ query_vec

    if exclude_image_id is not None:
        mask = metadata_df["image_id"].astype(str) == str(exclude_image_id)
        scores[mask.values] = -np.inf

    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = metadata_df.iloc[top_indices].copy()
    results["score"] = scores[top_indices]

    return results